In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path

print("V6-B — SEnNet-Inspired Architecture")
print("-" * 50)

for path in sorted(Path("/kaggle/input").rglob("*")):
    if path.is_file() and (
        path.name.endswith(".json")
        or path.name.endswith(".pth")
    ):
        print(path)

print("\nPASS: Available annotations and checkpoints listed.")

In [ ]:
import torch
import torchvision
from torchvision.models import (
    convnext_tiny,
    ConvNeXt_Tiny_Weights
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Load ImageNet-pretrained ConvNeXt-Tiny
encoder = convnext_tiny(
    weights=ConvNeXt_Tiny_Weights.DEFAULT
).to(device)

# Inspect the feature hierarchy
encoder.eval()

with torch.inference_mode():
    x = torch.randn(1, 3, 512, 512, device=device)

    features = []
    for i, block in enumerate(encoder.features):
        x = block(x)
        features.append(tuple(x.shape))

print("V6-B CONVNEXT ENCODER")
print("-" * 50)
print("Device:", device)
print("Feature shapes:")

for i, shape in enumerate(features):
    print(f"Stage {i}: {shape}")

print(
    "Encoder parameters:",
    f"{sum(p.numel() for p in encoder.parameters()):,}"
)

assert features[-1][-2:] == (16, 16)
print("\nPASS: Pretrained ConvNeXt encoder initialized.")

In [ ]:
import torch
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert device.type == "cuda", "Enable Kaggle GPU T4."

# Restore the ImageNet-pretrained ConvNeXt encoder
encoder = convnext_tiny(
    weights=ConvNeXt_Tiny_Weights.DEFAULT
).to(device)

# Use the ConvNeXtUNetV6B class defined in your previous cell
model_v6b = ConvNeXtUNetV6B(encoder).to(device)
model_v6b.eval()

with torch.inference_mode():
    x = torch.randn(1, 3, 512, 512, device=device)
    y = model_v6b(x)

print("Device:", device)
print("Output shape:", tuple(y.shape))
print("Parameters:", f"{sum(p.numel() for p in model_v6b.parameters()):,}")

assert y.shape == (1, 1, 512, 512)
print("PASS: V6-B ConvNeXt–U-Net initialized successfully.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = F.interpolate(
            x, size=skip.shape[-2:],
            mode="bilinear", align_corners=False
        )
        return self.conv(torch.cat([x, skip], dim=1))


class ConvNeXtUNetV6B(nn.Module):
    def __init__(self, pretrained_encoder):
        super().__init__()

        self.encoder = pretrained_encoder.features

        self.up3 = DecoderBlock(768, 384, 256)
        self.up2 = DecoderBlock(256, 192, 128)
        self.up1 = DecoderBlock(128, 96, 64)

        self.output_head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, x):
        original_size = x.shape[-2:]

        f1 = self.encoder[1](self.encoder[0](x))
        f2 = self.encoder[3](self.encoder[2](f1))
        f3 = self.encoder[5](self.encoder[4](f2))
        f4 = self.encoder[7](self.encoder[6](f3))

        x = self.up3(f4, f3)
        x = self.up2(x, f2)
        x = self.up1(x, f1)

        x = F.interpolate(
            x, size=original_size,
            mode="bilinear", align_corners=False
        )

        return self.output_head(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Enable Kaggle GPU T4 first."

model_v6b = ConvNeXtUNetV6B(encoder).to(device)
model_v6b.eval()

with torch.inference_mode():
    test = torch.randn(1, 3, 512, 512, device=device)
    output = model_v6b(test)

print("V6-B CONVNEXT–U-NET")
print("Device:", device)
print("Input:", tuple(test.shape))
print("Output:", tuple(output.shape))
print("Parameters:", f"{sum(p.numel() for p in model_v6b.parameters()):,}")

assert output.shape == (1, 1, 512, 512)
print("PASS: ConvNeXt–U-Net architecture ready.")

In [ ]:
import json
import numpy as np
import torch

from pathlib import Path
from PIL import Image, ImageDraw
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

root = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026/train"
)

image_dir = root / "train_images"
annotation_path = root / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

with open(annotation_path) as f:
    coco = json.load(f)

annotations_by_image = {}
for ann in coco["annotations"]:
    annotations_by_image.setdefault(ann["image_id"], []).append(ann)

# Deterministically select one expert record per physical photograph.
records_by_name = {}

for record in coco["images"]:
    name = record["file_name"]
    if not (image_dir / name).is_file():
        continue

    count = len(annotations_by_image.get(record["id"], []))
    previous = records_by_name.get(name)

    if previous is None or (
        count, -record["id"]
    ) > (
        len(annotations_by_image.get(previous["id"], [])),
        -previous["id"]
    ):
        records_by_name[name] = record

names = sorted(records_by_name)

train_names, val_names = train_test_split(
    names,
    test_size=0.2,
    random_state=42
)

class SolarFilamentDatasetV6B(Dataset):
    def __init__(self, names, samples_per_image=4):
        self.names = list(names)
        self.samples_per_image = samples_per_image

    def __len__(self):
        return len(self.names) * self.samples_per_image

    def __getitem__(self, idx):
        name = self.names[idx // self.samples_per_image]
        record = records_by_name[name]
        anns = annotations_by_image.get(record["id"], [])

        image = np.asarray(
            Image.open(image_dir / name).convert("RGB"),
            dtype=np.uint8
        )

        h, w = image.shape[:2]

        canvas = Image.new("L", (w, h), 0)
        draw = ImageDraw.Draw(canvas)

        for ann in anns:
            for polygon in ann.get("segmentation", []):
                points = list(zip(
                    polygon[0::2],
                    polygon[1::2]
                ))
                if len(points) >= 3:
                    draw.polygon(points, fill=1)

        mask = np.asarray(canvas, dtype=np.uint8)

        # Mix filament-centered and random crops.
        if anns and np.random.random() < 0.75:
            ann = anns[np.random.randint(len(anns))]
            x, y, bw, bh = ann["bbox"]
            cx = x + bw / 2
            cy = y + bh / 2

            left = int(np.clip(cx - 256, 0, w - 512))
            top = int(np.clip(cy - 256, 0, h - 512))
        else:
            left = np.random.randint(0, w - 512 + 1)
            top = np.random.randint(0, h - 512 + 1)

        image = image[top:top+512, left:left+512]
        mask = mask[top:top+512, left:left+512]

        image = torch.from_numpy(
            np.ascontiguousarray(image.transpose(2, 0, 1))
        ).float() / 255.0

        mask = torch.from_numpy(
            np.ascontiguousarray(mask[None])
        ).float()

        return image, mask


train_dataset_v6b = SolarFilamentDatasetV6B(train_names)

train_loader_v6b = DataLoader(
    train_dataset_v6b,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("V6-B DATASET")
print("-" * 45)
print("Unique photographs:", len(names))
print("Training photographs:", len(train_names))
print("Validation photographs:", len(val_names))
print("Training crops per epoch:", len(train_dataset_v6b))

assert len(train_names) == 565
assert len(val_names) == 142
assert not set(train_names) & set(val_names)

print("PASS: V6-B training dataset prepared.")

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# Select one expert annotation record per physical photograph.
# Use annotation count first, then string ID for deterministic ties.
records_by_name = {}

for record in coco["images"]:
    name = record["file_name"]

    if not (image_dir / name).is_file():
        continue

    previous = records_by_name.get(name)

    if previous is None:
        records_by_name[name] = record
        continue

    current_count = len(
        annotations_by_image.get(record["id"], [])
    )
    previous_count = len(
        annotations_by_image.get(previous["id"], [])
    )

    if (
        current_count > previous_count
        or (
            current_count == previous_count
            and str(record["id"]) < str(previous["id"])
        )
    ):
        records_by_name[name] = record

names = sorted(records_by_name)

train_names, val_names = train_test_split(
    names, test_size=0.2, random_state=42
)

# The previous cell failed before defining the dataset class.
class SolarFilamentDatasetV6B(torch.utils.data.Dataset):
    def __init__(self, names, samples_per_image=4):
        self.names = list(names)
        self.samples_per_image = samples_per_image

    def __len__(self):
        return len(self.names) * self.samples_per_image

    def __getitem__(self, idx):
        name = self.names[idx // self.samples_per_image]
        record = records_by_name[name]
        anns = annotations_by_image.get(record["id"], [])

        with Image.open(image_dir / name) as im:
            image = np.asarray(im.convert("RGB"), dtype=np.uint8)

        h, w = image.shape[:2]
        canvas = Image.new("L", (w, h), 0)
        draw = ImageDraw.Draw(canvas)

        for ann in anns:
            for polygon in ann.get("segmentation", []):
                points = list(zip(polygon[0::2], polygon[1::2]))
                if len(points) >= 3:
                    draw.polygon(points, fill=1)

        mask = np.asarray(canvas, dtype=np.uint8)

        if anns and np.random.random() < 0.75:
            ann = anns[np.random.randint(len(anns))]
            x, y, bw, bh = ann["bbox"]
            cx, cy = x + bw / 2, y + bh / 2
            left = int(np.clip(cx - 256, 0, w - 512))
            top = int(np.clip(cy - 256, 0, h - 512))
        else:
            left = np.random.randint(0, w - 511)
            top = np.random.randint(0, h - 511)

        image = image[top:top+512, left:left+512]
        mask = mask[top:top+512, left:left+512]

        image = torch.from_numpy(
            np.ascontiguousarray(image.transpose(2, 0, 1))
        ).float() / 255.0

        mask = torch.from_numpy(
            np.ascontiguousarray(mask[None])
        ).float()

        return image, mask


train_dataset_v6b = SolarFilamentDatasetV6B(train_names)

train_loader_v6b = DataLoader(
    train_dataset_v6b,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("V6-B DATASET")
print("-" * 45)
print("Unique photographs:", len(names))
print("Training photographs:", len(train_names))
print("Validation photographs:", len(val_names))
print("Training crops per epoch:", len(train_dataset_v6b))

assert len(names) == 707
assert len(train_names) == 565
assert len(val_names) == 142
assert not set(train_names) & set(val_names)

print("PASS: V6-B dataset prepared.")

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import time
from pathlib import Path

torch.manual_seed(2026)

device = torch.device("cuda")
model_v6b = model_v6b.to(device)
model_v6b.train()

optimizer = torch.optim.AdamW([
    {"params": model_v6b.encoder.parameters(), "lr": 1e-5},
    {"params": model_v6b.up3.parameters(), "lr": 1e-4},
    {"params": model_v6b.up2.parameters(), "lr": 1e-4},
    {"params": model_v6b.up1.parameters(), "lr": 1e-4},
    {"params": model_v6b.output_head.parameters(), "lr": 1e-4},
], weight_decay=1e-4)

scaler = torch.amp.GradScaler("cuda")

def segmentation_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(
        logits, targets
    )

    probabilities = torch.sigmoid(logits)

    dims = (1, 2, 3)
    intersection = (probabilities * targets).sum(dim=dims)
    denominator = (
        probabilities.sum(dim=dims) + targets.sum(dim=dims)
    )

    dice = 1 - (
        (2 * intersection + 1)
        / (denominator + 1)
    ).mean()

    return bce + dice

losses = []
start = time.time()

print("V6-B CONVNEXT–U-NET TRAINING")
print("-" * 50)

for step, (images, masks) in enumerate(
    train_loader_v6b, start=1
):
    images = images.to(device, non_blocking=True)
    masks = masks.to(device, non_blocking=True)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast("cuda", dtype=torch.float16):
        logits = model_v6b(images)
        loss = segmentation_loss(logits, masks)

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at step {step}")

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    losses.append(float(loss.item()))

    if step % 200 == 0:
        print(
            f"Step {step:04d}/{len(train_loader_v6b)} | "
            f"Mean loss: {np.mean(losses[-200:]):.4f}"
        )

checkpoint = Path(
    "/kaggle/working/v6b_convnext_unet_epoch1.pth"
)

torch.save(model_v6b.state_dict(), checkpoint)

print("\nTRAINING SUMMARY")
print("-" * 50)
print("Steps:", len(losses))
print("First 100-step mean:", round(np.mean(losses[:100]), 4))
print("Last 100-step mean:", round(np.mean(losses[-100:]), 4))
print("Time:", round((time.time() - start) / 60, 2), "minutes")
print("Checkpoint:", checkpoint)

print("\nPASS: V6-B first training epoch completed.")

In [ ]:
import numpy as np
import torch
from PIL import Image, ImageDraw

model_v6b.eval()

rng = np.random.default_rng(2026)
selected = rng.choice(
    val_names,
    size=100,
    replace=False
)

total_intersection = 0
total_predicted = 0
total_ground_truth = 0

image_dice = []
image_iou = []

with torch.inference_mode():
    for name in selected:
        record = records_by_name[name]
        anns = annotations_by_image.get(record["id"], [])

        with Image.open(image_dir / name) as im:
            rgb = np.asarray(im.convert("RGB"), dtype=np.uint8)

        h, w = rgb.shape[:2]

        canvas = Image.new("L", (w, h), 0)
        draw = ImageDraw.Draw(canvas)

        for ann in anns:
            for polygon in ann.get("segmentation", []):
                points = list(zip(
                    polygon[0::2],
                    polygon[1::2]
                ))
                if len(points) >= 3:
                    draw.polygon(points, fill=1)

        gt_full = np.asarray(canvas, dtype=bool)

        # Deterministic filament-centered crop
        if anns:
            ann = max(
                anns,
                key=lambda a: float(a.get("area", 0))
            )
            x, y, bw, bh = ann["bbox"]

            left = int(np.clip(
                x + bw / 2 - 256, 0, w - 512
            ))
            top = int(np.clip(
                y + bh / 2 - 256, 0, h - 512
            ))
        else:
            left = (w - 512) // 2
            top = (h - 512) // 2

        crop = rgb[top:top+512, left:left+512]
        gt = gt_full[top:top+512, left:left+512]

        tensor = torch.from_numpy(
            np.ascontiguousarray(
                crop.transpose(2, 0, 1)
            )
        ).unsqueeze(0).float().to(device) / 255.0

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model_v6b(tensor)

        pred = (
            torch.sigmoid(logits)[0, 0].float().cpu().numpy()
            >= 0.5
        )

        intersection = np.logical_and(pred, gt).sum()
        pred_area = pred.sum()
        gt_area = gt.sum()
        union = pred_area + gt_area - intersection

        dice = (
            2 * intersection / (pred_area + gt_area)
            if pred_area + gt_area > 0 else 1.0
        )

        iou = intersection / union if union > 0 else 1.0

        image_dice.append(dice)
        image_iou.append(iou)

        total_intersection += intersection
        total_predicted += pred_area
        total_ground_truth += gt_area

micro_dice = (
    2 * total_intersection /
    (total_predicted + total_ground_truth)
)

print("V6-B HELD-OUT VALIDATION")
print("-" * 50)
print("Photographs evaluated:", len(selected))
print("Mean crop Dice:", round(float(np.mean(image_dice)), 4))
print("Mean crop IoU:", round(float(np.mean(image_iou)), 4))
print("Global pixel Dice:", round(float(micro_dice), 4))
print("PASS: V6-B validation completed.")

In [ ]:
import numpy as np
import torch
import cv2
from PIL import Image, ImageDraw
from scipy.optimize import linear_sum_assignment

model_v6b.eval()

# Same 10 held-out photographs previously used for V6-A
evaluation_names = [
    "20110421164914Mh.jpeg",
    "20220718185352Mh.jpeg",
    "20111226155454Bh.jpeg",
    "20141007224934Lh.jpeg",
    "20160726135354Bh.jpeg",
    "20130502063134Lh.jpeg",
    "20110823120334Ch.jpeg",
    "20140116180214Mh.jpeg",
    "20130510134654Bh.jpeg",
    "20150112081954Uh.jpeg"
]

assert set(evaluation_names).issubset(set(val_names))

def tile_positions(length, tile=512, stride=256):
    p = list(range(0, length - tile + 1, stride))
    if p[-1] != length - tile:
        p.append(length - tile)
    return p

def predict_v6b_instances(name):
    with Image.open(image_dir / name) as im:
        rgb = np.asarray(im.convert("RGB"), dtype=np.uint8)

    h, w = rgb.shape[:2]
    probability_sum = np.zeros((h, w), dtype=np.float32)
    count = np.zeros((h, w), dtype=np.float32)

    # Batched overlapping-tile inference
    crops, locations = [], []

    with torch.inference_mode():
        for top in tile_positions(h):
            for left in tile_positions(w):
                crop = rgb[top:top+512, left:left+512]

                crops.append(crop.transpose(2, 0, 1))
                locations.append((left, top))

                if len(crops) == 4:
                    batch = torch.from_numpy(
                        np.ascontiguousarray(np.stack(crops))
                    ).float().to(device) / 255.0

                    with torch.autocast("cuda", dtype=torch.float16):
                        probs = torch.sigmoid(model_v6b(batch))

                    probs = probs[:, 0].float().cpu().numpy()

                    for prob, (x, y) in zip(probs, locations):
                        probability_sum[y:y+512, x:x+512] += prob
                        count[y:y+512, x:x+512] += 1

                    crops, locations = [], []

        if crops:
            batch = torch.from_numpy(
                np.ascontiguousarray(np.stack(crops))
            ).float().to(device) / 255.0

            with torch.autocast("cuda", dtype=torch.float16):
                probs = torch.sigmoid(model_v6b(batch))

            probs = probs[:, 0].float().cpu().numpy()

            for prob, (x, y) in zip(probs, locations):
                probability_sum[y:y+512, x:x+512] += prob
                count[y:y+512, x:x+512] += 1

    prob_full = probability_sum / np.maximum(count, 1)

    # Fixed preliminary postprocessing, analogous to V5-B
    binary = (prob_full >= 0.55).astype(np.uint8)

    kernel = np.ones((3, 3), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )

    instances = []

    for i in range(1, n):
        area = int(stats[i, cv2.CC_STAT_AREA])

        if area < 800:
            continue

        component = labels == i

        if float(prob_full[component].mean()) < 0.80:
            continue

        instances.append(component)

    return instances

def gt_instances_v6b(name):
    record = records_by_name[name]
    anns = annotations_by_image.get(record["id"], [])
    masks = []

    for ann in anns:
        canvas = Image.new("1", (2048, 2048), 0)
        draw = ImageDraw.Draw(canvas)

        for poly in ann.get("segmentation", []):
            pts = list(zip(poly[0::2], poly[1::2]))
            if len(pts) >= 3:
                draw.polygon(pts, fill=1)

        mask = np.asarray(canvas, dtype=bool)
        if mask.any():
            masks.append(mask)

    return masks

def evaluate_instances_v6b(pred, gt):
    matrix = np.zeros((len(pred), len(gt)), dtype=np.float32)

    for i, p in enumerate(pred):
        p_area = p.sum()
        for j, g in enumerate(gt):
            inter = np.logical_and(p, g).sum()
            union = p_area + g.sum() - inter
            matrix[i, j] = inter / union if union else 0.0

    valid = matrix >= 0.5

    rows, cols = linear_sum_assignment(
        -(valid.astype(np.float64) * 2 + matrix)
    )

    matched = [
        float(matrix[r, c])
        for r, c in zip(rows, cols)
        if valid[r, c]
    ]

    tp = len(matched)
    return np.array(
        [tp, len(pred)-tp, len(gt)-tp, sum(matched)],
        dtype=float
    )

totals = np.zeros(4, dtype=float)

print("V6-B FULL-RESOLUTION VALIDATION")
print("-" * 70)

for idx, name in enumerate(evaluation_names, 1):
    pred = predict_v6b_instances(name)
    gt = gt_instances_v6b(name)

    scores = evaluate_instances_v6b(pred, gt)
    totals += scores

    print(
        f"{idx:02d}/10 | {name} | "
        f"GT={len(gt)} Pred={len(pred)} TP={int(scores[0])}"
    )

tp, fp, fn, matched_iou = totals
denominator = tp + 0.5 * fp + 0.5 * fn
pq = matched_iou / denominator if denominator else 0.0

print("\nAGGREGATED V6-B RESULTS")
print("-" * 50)
print("TP:", int(tp))
print("FP:", int(fp))
print("FN:", int(fn))
print("PQ:", round(pq, 4))
print("V6-A merged reference PQ: 0.1998")

print("\nPASS: V6-B full-resolution evaluation completed.")

In [ ]:
import numpy as np
import torch
import cv2
from PIL import Image

model_v6b.eval()
probability_cache_v6b = {}

with torch.inference_mode():
    for index, name in enumerate(evaluation_names, 1):
        with Image.open(image_dir / name) as im:
            rgb = np.asarray(im.convert("RGB"), dtype=np.uint8)

        h, w = rgb.shape[:2]
        prob_sum = np.zeros((h, w), dtype=np.float32)
        count = np.zeros((h, w), dtype=np.uint8)

        crops, locations = [], []

        def flush_batch():
            if not crops:
                return

            batch = torch.from_numpy(
                np.ascontiguousarray(np.stack(crops))
            ).float().to(device) / 255.0

            with torch.autocast("cuda", dtype=torch.float16):
                probs = torch.sigmoid(model_v6b(batch))

            probs = probs[:, 0].float().cpu().numpy()

            for prob, (left, top) in zip(probs, locations):
                prob_sum[top:top+512, left:left+512] += prob
                count[top:top+512, left:left+512] += 1

            crops.clear()
            locations.clear()

        for top in tile_positions(h):
            for left in tile_positions(w):
                crop = rgb[top:top+512, left:left+512]
                crops.append(crop.transpose(2, 0, 1))
                locations.append((left, top))

                if len(crops) == 4:
                    flush_batch()

        flush_batch()

        probability_cache_v6b[name] = (
            prob_sum / np.maximum(count, 1)
        )

        print(f"Cached {index}/10: {name}")

# Evaluate without additional neural-network inference
gt_cache = {
    name: gt_instances_v6b(name)
    for name in evaluation_names
}

print("\nMINIMUM COMPONENT AREA COMPARISON")
print("-" * 55)
print("Min area | TP | FP | FN | PQ")

for min_area in [100, 300, 500, 800, 1200]:
    totals = np.zeros(4, dtype=float)

    for name in evaluation_names:
        prob = probability_cache_v6b[name]

        binary = (prob >= 0.55).astype(np.uint8)
        kernel = np.ones((3, 3), np.uint8)

        binary = cv2.morphologyEx(
            binary, cv2.MORPH_OPEN, kernel
        )
        binary = cv2.morphologyEx(
            binary, cv2.MORPH_CLOSE, kernel
        )

        n, labels, stats, _ = cv2.connectedComponentsWithStats(
            binary, connectivity=8
        )

        predictions = []

        for i in range(1, n):
            area = int(stats[i, cv2.CC_STAT_AREA])

            if area < min_area:
                continue

            component = labels == i

            if float(prob[component].mean()) < 0.80:
                continue

            predictions.append(component)

        totals += evaluate_instances_v6b(
            predictions, gt_cache[name]
        )

    tp, fp, fn, matched_iou = totals
    denominator = tp + 0.5 * fp + 0.5 * fn
    pq = matched_iou / denominator if denominator else 0.0

    print(
        f"{min_area:8d} | "
        f"{int(tp):2d} | {int(fp):2d} | "
        f"{int(fn):2d} | {pq:.4f}"
    )

print("\nPASS: Instance extraction comparison completed.")

In [ ]:
import torch
import numpy as np
import time
from pathlib import Path

model_v6b.train()

additional_epochs = 2
epoch_losses = []
start = time.time()

for epoch in range(2, 2 + additional_epochs):
    losses = []

    for images, masks in train_loader_v6b:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model_v6b(images)
            loss = segmentation_loss(logits, masks)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss in epoch {epoch}"
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        losses.append(float(loss.item()))

    mean_loss = float(np.mean(losses))
    epoch_losses.append(mean_loss)

    print(
        f"Epoch {epoch}/3 | "
        f"Mean training loss: {mean_loss:.4f}"
    )

checkpoint = Path(
    "/kaggle/working/v6b_convnext_unet_epoch3.pth"
)

torch.save(model_v6b.state_dict(), checkpoint)

print("\nTRAINING SUMMARY")
print("-" * 45)
print("Additional epochs:", additional_epochs)
print("Final epoch loss:", round(epoch_losses[-1], 4))
print("Training time:", round((time.time()-start)/60, 2), "minutes")
print("Checkpoint:", checkpoint)
print("PASS: V6-B three-epoch training completed.")

In [ ]:
import numpy as np
import torch
import cv2
from PIL import Image

model_v6b.eval()

totals = np.zeros(4, dtype=float)

MIN_AREA = 500

print("V6-B — THREE-EPOCH FULL-IMAGE EVALUATION")
print("-" * 65)

for idx, name in enumerate(evaluation_names, 1):

    # Generate probabilities with the updated model
    with Image.open(image_dir / name) as im:
        rgb = np.asarray(im.convert("RGB"), dtype=np.uint8)

    h, w = rgb.shape[:2]
    prob_sum = np.zeros((h, w), dtype=np.float32)
    count = np.zeros((h, w), dtype=np.uint8)

    crops, locations = [], []

    def process_batch():
        if not crops:
            return

        batch = torch.from_numpy(
            np.ascontiguousarray(np.stack(crops))
        ).float().to(device) / 255.0

        with torch.inference_mode(), torch.autocast(
            "cuda", dtype=torch.float16
        ):
            probs = torch.sigmoid(model_v6b(batch))

        probs = probs[:, 0].float().cpu().numpy()

        for p, (left, top) in zip(probs, locations):
            prob_sum[top:top+512, left:left+512] += p
            count[top:top+512, left:left+512] += 1

        crops.clear()
        locations.clear()

    for top in tile_positions(h):
        for left in tile_positions(w):
            crop = rgb[top:top+512, left:left+512]

            crops.append(crop.transpose(2, 0, 1))
            locations.append((left, top))

            if len(crops) == 4:
                process_batch()

    process_batch()

    prob = prob_sum / np.maximum(count, 1)

    binary = (prob >= 0.55).astype(np.uint8)

    kernel = np.ones((3, 3), np.uint8)

    binary = cv2.morphologyEx(
        binary, cv2.MORPH_OPEN, kernel
    )

    binary = cv2.morphologyEx(
        binary, cv2.MORPH_CLOSE, kernel
    )

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )

    predictions = []

    for i in range(1, n):
        area = int(stats[i, cv2.CC_STAT_AREA])

        if area < MIN_AREA:
            continue

        component = labels == i

        if float(prob[component].mean()) < 0.80:
            continue

        predictions.append(component)

    gt = gt_cache[name]

    scores = evaluate_instances_v6b(predictions, gt)
    totals += scores

    print(
        f"{idx:02d}/10 | {name} | "
        f"GT={len(gt)} Pred={len(predictions)} "
        f"TP={int(scores[0])}"
    )

tp, fp, fn, matched_iou = totals

denominator = tp + 0.5 * fp + 0.5 * fn
pq = matched_iou / denominator if denominator else 0.0

print("\nAGGREGATED RESULTS")
print("-" * 50)
print("TP:", int(tp))
print("FP:", int(fp))
print("FN:", int(fn))
print("PQ:", round(pq, 4))
print("Epoch 1 reference PQ (500px): 0.2738")

print("\nPASS: Three-epoch evaluation completed.")

In [ ]:
import torch
import numpy as np
from pathlib import Path

model_v6b.train()

for epoch in [4, 5]:
    losses = []

    for images, masks in train_loader_v6b:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model_v6b(images)
            loss = segmentation_loss(logits, masks)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at epoch {epoch}")

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        losses.append(loss.item())

    print(
        f"Epoch {epoch}/5 | "
        f"Mean loss: {np.mean(losses):.4f}"
    )

checkpoint = Path(
    "/kaggle/working/v6b_convnext_unet_epoch5.pth"
)

torch.save(model_v6b.state_dict(), checkpoint)

print("\nSaved:", checkpoint)
print("PASS: Five-epoch training completed.")

In [ ]:
import numpy as np
import torch
import cv2
from PIL import Image

model_v6b.eval()
totals = np.zeros(4, dtype=float)

with torch.inference_mode():
    for idx, name in enumerate(evaluation_names, 1):

        with Image.open(image_dir / name) as im:
            rgb = np.asarray(im.convert("RGB"), dtype=np.uint8)

        h, w = rgb.shape[:2]
        prob_sum = np.zeros((h, w), dtype=np.float32)
        count = np.zeros((h, w), dtype=np.uint8)

        crops, locations = [], []

        def process_batch():
            if not crops:
                return

            batch = torch.from_numpy(
                np.ascontiguousarray(np.stack(crops))
            ).float().to(device) / 255.0

            with torch.autocast("cuda", dtype=torch.float16):
                probs = torch.sigmoid(model_v6b(batch))

            probs = probs[:, 0].float().cpu().numpy()

            for p, (left, top) in zip(probs, locations):
                prob_sum[top:top+512, left:left+512] += p
                count[top:top+512, left:left+512] += 1

            crops.clear()
            locations.clear()

        for top in tile_positions(h):
            for left in tile_positions(w):
                crop = rgb[top:top+512, left:left+512]
                crops.append(crop.transpose(2, 0, 1))
                locations.append((left, top))

                if len(crops) == 4:
                    process_batch()

        process_batch()

        prob = prob_sum / np.maximum(count, 1)

        binary = (prob >= 0.55).astype(np.uint8)
        kernel = np.ones((3, 3), np.uint8)

        binary = cv2.morphologyEx(
            binary, cv2.MORPH_OPEN, kernel
        )
        binary = cv2.morphologyEx(
            binary, cv2.MORPH_CLOSE, kernel
        )

        n, labels, stats, _ = cv2.connectedComponentsWithStats(
            binary, connectivity=8
        )

        predictions = []

        for i in range(1, n):
            area = int(stats[i, cv2.CC_STAT_AREA])

            if area < 500:
                continue

            component = labels == i

            if float(prob[component].mean()) < 0.80:
                continue

            predictions.append(component)

        scores = evaluate_instances_v6b(
            predictions, gt_cache[name]
        )

        totals += scores

        print(
            f"{idx:02d}/10 | GT={len(gt_cache[name])} | "
            f"Pred={len(predictions)} | TP={int(scores[0])}"
        )

tp, fp, fn, matched_iou = totals
pq = matched_iou / (tp + 0.5 * fp + 0.5 * fn)

print("\nEPOCH 5 — FULL-RESOLUTION RESULTS")
print("-" * 45)
print("TP:", int(tp))
print("FP:", int(fp))
print("FN:", int(fn))
print("PQ:", round(float(pq), 4))
print("Epoch 3 reference PQ: 0.3302")
print("PQ change:", round(float(pq - 0.3302), 4))

print("\nPASS: Epoch-5 evaluation completed.")

In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageDraw

# Restore the frozen epoch-3 model, even if epoch 5
# was the last model loaded in notebook memory.
checkpoint_path = Path(
    "/kaggle/working/v6b_final_research/"
    "v6b_convnext_unet_best.pth"
)

model_v6b.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
)
model_v6b.eval()

# Start with an image containing many expert filaments
name = "20141007224934Lh.jpeg"

# Reuse our existing full-resolution inference function
# from the V6-B notebook, where available.
rgb = np.asarray(
    Image.open(image_dir / name).convert("RGB")
)

gt_masks = gt_instances_v6b(name)
gt_union = np.logical_or.reduce(gt_masks)

# Full-image probability prediction using overlapping tiles
H, W = rgb.shape[:2]
prob_sum = np.zeros((H, W), dtype=np.float32)
count = np.zeros((H, W), dtype=np.float32)

with torch.inference_mode():
    for top in tile_positions(H):
        for left in tile_positions(W):
            crop = rgb[top:top+512, left:left+512]

            x = torch.from_numpy(
                np.ascontiguousarray(
                    crop.transpose(2, 0, 1)
                )
            ).unsqueeze(0).float().to(device) / 255.0

            with torch.autocast("cuda", dtype=torch.float16):
                p = torch.sigmoid(model_v6b(x))

            p = p[0, 0].float().cpu().numpy()

            prob_sum[top:top+512, left:left+512] += p
            count[top:top+512, left:left+512] += 1

prob = prob_sum / np.maximum(count, 1)

# Apply exactly the same postprocessing as before
binary = (prob >= 0.55).astype(np.uint8)
kernel = np.ones((3, 3), dtype=np.uint8)

clean = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
clean = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, kernel)

n, labels, stats, _ = cv2.connectedComponentsWithStats(
    clean, connectivity=8
)

final_mask = np.zeros((H, W), dtype=bool)

for i in range(1, n):
    component = labels == i

    if stats[i, cv2.CC_STAT_AREA] < 500:
        continue

    if float(prob[component].mean()) < 0.80:
        continue

    final_mask |= component

fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0, 0].imshow(rgb)
axes[0, 0].set_title("A — Original solar image")

axes[0, 1].imshow(gt_union, cmap="gray")
axes[0, 1].set_title(f"B — Expert masks ({len(gt_masks)} filaments)")

im = axes[1, 0].imshow(
    prob, cmap="inferno", vmin=0, vmax=1
)
axes[1, 0].set_title("C — Raw V6-B probability map")
fig.colorbar(im, ax=axes[1, 0], fraction=0.046)

axes[1, 1].imshow(final_mask, cmap="gray")
axes[1, 1].set_title("D — Final post-processed mask")

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Expert instances:", len(gt_masks))
print("GT filament pixels:", int(gt_union.sum()))
print("Raw thresholded pixels:", int(binary.sum()))
print("After morphology:", int(clean.sum()))
print("Final retained pixels:", int(final_mask.sum()))
print("PASS: Prediction stages visualized.")

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

working = Path("/kaggle/working")

source = working / "v6b_convnext_unet_epoch3.pth"
assert source.is_file(), "Epoch-3 checkpoint missing."

folder = working / "v6b_final_research"
folder.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    source,
    folder / "v6b_convnext_unet_best.pth"
)

metadata = {
    "architecture": "V6-B ConvNeXt-Tiny U-Net",
    "encoder": "ImageNet-pretrained ConvNeXt-Tiny",
    "selected_epoch": 3,
    "train_photographs": 565,
    "validation_photographs": 142,
    "evaluated_full_images": 10,
    "ground_truth_instances": 92,
    "postprocessing": {
        "probability_threshold": 0.55,
        "minimum_area": 500,
        "minimum_component_confidence": 0.80
    },
    "validation": {
        "TP": 49,
        "FP": 53,
        "FN": 43,
        "PQ": 0.3302
    },
    "limitations": (
        "Results are from 10 selected validation photographs; "
        "not an official Kaggle score. Postprocessing was "
        "explored on the same photographs."
    )
}

with open(folder / "experiment.json", "w") as f:
    json.dump(metadata, f, indent=2)

archive = working / "v6b_final_research.zip"

with zipfile.ZipFile(
    archive, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for path in folder.iterdir():
        z.write(path, arcname=path.name)

print("V6-B FINAL CHECKPOINT")
print("-" * 45)
print("Selected epoch:", metadata["selected_epoch"])
print("Validation PQ:", metadata["validation"]["PQ"])
print("Checkpoint:", folder / "v6b_convnext_unet_best.pth")
print("Archive:", archive)
print("Archive size:", round(archive.stat().st_size / 1024**2, 2), "MB")
print("PASS: V6-B research checkpoint frozen.")

In [ ]:
from pathlib import Path

print("V6-B CHECKPOINT RECOVERY")
print("-" * 50)

matches = []

search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input")
]

for root in search_roots:
    if root.exists():
        matches.extend(root.rglob("v6b_convnext_unet_best.pth"))
        matches.extend(root.rglob("v6b_convnext_unet_epoch3.pth"))
        matches.extend(root.rglob("v6b_final_research.zip"))

matches = sorted(set(matches))

if matches:
    for path in matches:
        print("FOUND:", path)
        print("Size:", round(path.stat().st_size / 1024**2, 2), "MB")
else:
    print("No saved checkpoint found in the current session.")

print("\nSearch completed.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import convnext_tiny
from pathlib import Path

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels + skip_channels,
                out_channels, 3, padding=1, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels, out_channels,
                3, padding=1, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False
        )
        return self.conv(torch.cat([x, skip], dim=1))


class ConvNeXtUNetV6B(nn.Module):
    def __init__(self, backbone):
        super().__init__()

        self.encoder = backbone.features
        self.up3 = DecoderBlock(768, 384, 256)
        self.up2 = DecoderBlock(256, 192, 128)
        self.up1 = DecoderBlock(128, 96, 64)

        self.output_head = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, x):
        size = x.shape[-2:]

        f1 = self.encoder[1](self.encoder[0](x))
        f2 = self.encoder[3](self.encoder[2](f1))
        f3 = self.encoder[5](self.encoder[4](f2))
        f4 = self.encoder[7](self.encoder[6](f3))

        x = self.up3(f4, f3)
        x = self.up2(x, f2)
        x = self.up1(x, f1)

        x = F.interpolate(
            x, size=size,
            mode="bilinear",
            align_corners=False
        )

        return self.output_head(x)


checkpoint = Path(
    "/kaggle/working/v6b_final_research/"
    "v6b_convnext_unet_best.pth"
)

assert checkpoint.is_file(), "Checkpoint not found."

# The checkpoint contains the trained encoder and decoder,
# so downloading ImageNet weights again is unnecessary.
backbone = convnext_tiny(weights=None)

model_v6b = ConvNeXtUNetV6B(backbone).to(device)

state = torch.load(
    checkpoint,
    map_location="cpu",
    weights_only=True
)

model_v6b.load_state_dict(state, strict=True)
model_v6b.eval()

print("V6-B MODEL RESTORATION")
print("-" * 45)
print("Device:", device)
print("Checkpoint:", checkpoint.name)
print(
    "Parameters:",
    f"{sum(p.numel() for p in model_v6b.parameters()):,}"
)
print("Frozen model restored: True")

assert sum(
    p.numel() for p in model_v6b.parameters()
) == 31912353

print("\nPASS: V6-B restored without retraining.")

In [ ]:
import json
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageDraw

root = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026/train"
)
image_dir = root / "train_images"

with open(root / "MAGFiLO_1.0_Annotations_kaggle2026_train.json") as f:
    coco = json.load(f)

name = "20141007224934Lh.jpeg"

annotations_by_id = {}
for a in coco["annotations"]:
    annotations_by_id.setdefault(a["image_id"], []).append(a)

records = [r for r in coco["images"] if r["file_name"] == name]

# Same annotation-selection policy as V6-B
record = sorted(
    records,
    key=lambda r: (
        -len(annotations_by_id.get(r["id"], [])),
        str(r["id"])
    )
)[0]

with Image.open(image_dir / name) as im:
    rgb = np.asarray(im.convert("RGB"))

H, W = rgb.shape[:2]
gt_union = np.zeros((H, W), dtype=bool)
gt_masks = []

for ann in annotations_by_id.get(record["id"], []):
    canvas = Image.new("L", (W, H), 0)
    draw = ImageDraw.Draw(canvas)

    for poly in ann.get("segmentation", []):
        pts = list(zip(poly[0::2], poly[1::2]))
        if len(pts) >= 3:
            draw.polygon(pts, fill=1)

    mask = np.asarray(canvas, dtype=bool)
    if mask.any():
        gt_masks.append(mask)
        gt_union |= mask

# Full-image inference, 512px tiles with 256px stride
prob_sum = np.zeros((H, W), dtype=np.float32)
counts = np.zeros((H, W), dtype=np.float32)

model_v6b.eval()

with torch.inference_mode():
    for top in range(0, H - 511, 256):
        for left in range(0, W - 511, 256):
            crop = rgb[top:top+512, left:left+512]

            x = torch.from_numpy(
                np.ascontiguousarray(crop.transpose(2, 0, 1))
            ).unsqueeze(0).float().to(device) / 255.0

            with torch.autocast("cuda", dtype=torch.float16):
                p = torch.sigmoid(model_v6b(x))

            p = p[0, 0].float().cpu().numpy()

            prob_sum[top:top+512, left:left+512] += p
            counts[top:top+512, left:left+512] += 1

prob = prob_sum / np.maximum(counts, 1)

binary = (prob >= 0.55).astype(np.uint8)
kernel = np.ones((3, 3), np.uint8)

opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
clean = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)

n, labels, stats, _ = cv2.connectedComponentsWithStats(
    clean, connectivity=8
)

final_mask = np.zeros((H, W), dtype=bool)
retained = 0

for i in range(1, n):
    component = labels == i

    if stats[i, cv2.CC_STAT_AREA] < 500:
        continue

    if float(prob[component].mean()) < 0.80:
        continue

    final_mask |= component
    retained += 1

fig, axes = plt.subplots(2, 2, figsize=(12, 12))

axes[0, 0].imshow(rgb)
axes[0, 0].set_title("Original solar image")

axes[0, 1].imshow(gt_union, cmap="gray")
axes[0, 1].set_title(f"Expert annotations: {len(gt_masks)} instances")

im = axes[1, 0].imshow(prob, cmap="inferno", vmin=0, vmax=1)
axes[1, 0].set_title("Raw neural-network probabilities")
fig.colorbar(im, ax=axes[1, 0], fraction=0.046)

axes[1, 1].imshow(final_mask, cmap="gray")
axes[1, 1].set_title(f"Post-processed prediction: {retained} instances")

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("GT instances:", len(gt_masks))
print("Thresholded pixels:", int(binary.sum()))
print("After opening:", int(opened.sum()))
print("After closing:", int(clean.sum()))
print("Final retained pixels:", int(final_mask.sum()))
print("Retained instances:", retained)
print("PASS: Frozen V6-B failure-analysis image generated.")

In [ ]:
import numpy as np

print("V6-B — PER-FILAMENT COVERAGE ANALYSIS")
print("-" * 77)
print(
    "GT | Area(px) | Mean prob | Threshold | Morphology | Final | Lost pixels"
)

results = []

for i, gt in enumerate(gt_masks, start=1):
    area = int(gt.sum())

    mean_prob = float(prob[gt].mean())

    raw_coverage = float(
        np.logical_and(gt, binary > 0).sum() / area
    )

    morph_coverage = float(
        np.logical_and(gt, clean > 0).sum() / area
    )

    final_coverage = float(
        np.logical_and(gt, final_mask).sum() / area
    )

    lost_pixels = int(
        np.logical_and(gt, ~final_mask).sum()
    )

    results.append({
        "id": i,
        "area": area,
        "mean_prob": mean_prob,
        "raw": raw_coverage,
        "morph": morph_coverage,
        "final": final_coverage,
        "lost": lost_pixels
    })

    print(
        f"{i:2d} | "
        f"{area:8d} | "
        f"{mean_prob:9.3f} | "
        f"{raw_coverage:9.1%} | "
        f"{morph_coverage:10.1%} | "
        f"{final_coverage:5.1%} | "
        f"{lost_pixels:6d}"
    )

print("\nSUMMARY")
print(
    "Filaments with >= 70% final pixel coverage:",
    sum(r["final"] >= 0.70 for r in results),
    "/", len(results)
)

print(
    "Filaments losing >= 20 percentage points "
    "during postprocessing:",
    sum(r["raw"] - r["final"] >= 0.20 for r in results)
)

print("PASS: Per-filament coverage analysis completed.")

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

# Reconstruct the same retained instance masks
pred_instances = []

for i in range(1, n):
    component = labels == i

    if stats[i, cv2.CC_STAT_AREA] < 500:
        continue

    if float(prob[component].mean()) < 0.80:
        continue

    pred_instances.append(component)

# Pairwise instance IoU
iou_matrix = np.zeros(
    (len(gt_masks), len(pred_instances)),
    dtype=np.float32
)

# Pixel recall of each GT using its single best prediction
best_recall = np.zeros(len(gt_masks), dtype=np.float32)

for i, gt in enumerate(gt_masks):
    gt_area = gt.sum()

    for j, pred in enumerate(pred_instances):
        intersection = np.logical_and(gt, pred).sum()
        union = np.logical_or(gt, pred).sum()

        iou_matrix[i, j] = (
            intersection / union if union else 0.0
        )

        best_recall[i] = max(
            best_recall[i],
            intersection / gt_area
        )

print("V6-B — INSTANCE-LEVEL FAILURE DIAGNOSTIC")
print("-" * 72)
print("GT | Union coverage | Best-instance recall | Best IoU | Status")

for i, gt in enumerate(gt_masks):
    union_coverage = results[i]["final"]
    best_iou = float(iou_matrix[i].max()) if len(pred_instances) else 0

    status = (
        "MATCH CANDIDATE" if best_iou >= 0.5
        else "HIGH COVERAGE, POOR MATCH"
        if union_coverage >= 0.70
        else "LOW COVERAGE"
    )

    print(
        f"{i+1:2d} | "
        f"{union_coverage:13.1%} | "
        f"{best_recall[i]:20.1%} | "
        f"{best_iou:8.3f} | {status}"
    )

print("\nGT instances:", len(gt_masks))
print("Predicted instances:", len(pred_instances))
print(
    "High coverage but no IoU >= 0.5:",
    sum(
        results[i]["final"] >= 0.70
        and iou_matrix[i].max() < 0.5
        for i in range(len(gt_masks))
    )
)

print("PASS: Instance-level failure diagnostic completed.")

In [ ]:
import numpy as np
import cv2
from scipy.optimize import linear_sum_assignment

def evaluate_threshold(threshold):
    binary_t = (prob >= threshold).astype(np.uint8)
    kernel = np.ones((3, 3), dtype=np.uint8)

    cleaned = cv2.morphologyEx(
        binary_t, cv2.MORPH_OPEN, kernel
    )
    cleaned = cv2.morphologyEx(
        cleaned, cv2.MORPH_CLOSE, kernel
    )

    count, lab, st, _ = cv2.connectedComponentsWithStats(
        cleaned, connectivity=8
    )

    predictions = []

    for j in range(1, count):
        if st[j, cv2.CC_STAT_AREA] < 500:
            continue

        mask = lab == j

        if float(prob[mask].mean()) < 0.80:
            continue

        predictions.append(mask)

    ious = np.zeros(
        (len(gt_masks), len(predictions)), dtype=np.float32
    )

    for i, gt in enumerate(gt_masks):
        for j, pred in enumerate(predictions):
            intersection = np.logical_and(gt, pred).sum()
            union = np.logical_or(gt, pred).sum()

            if union:
                ious[i, j] = intersection / union

    if ious.size:
        rows, cols = linear_sum_assignment(-ious)
        matched = [
            float(ious[i, j])
            for i, j in zip(rows, cols)
            if ious[i, j] >= 0.50
        ]
    else:
        matched = []

    tp = len(matched)
    fp = len(predictions) - tp
    fn = len(gt_masks) - tp

    pq = sum(matched) / (tp + 0.5 * fp + 0.5 * fn)

    return tp, fp, fn, pq


print("V6-B — THRESHOLD ABLATION")
print("-" * 47)
print("Threshold | TP | FP | FN | PQ")
print("-" * 47)

for threshold in [0.35, 0.45, 0.55, 0.65, 0.75]:
    tp, fp, fn, pq = evaluate_threshold(threshold)

    marker = "  <-- original" if threshold == 0.55 else ""

    print(
        f"   {threshold:.2f}   | "
        f"{tp:2d} | {fp:2d} | {fn:2d} | "
        f"{pq:.4f}{marker}"
    )

print("\nPASS: Threshold experiment completed.")

In [ ]:
print("V6-B EVALUATION ENVIRONMENT")
print("-" * 45)

for key in [
    "evaluation_names",
    "val_names",
    "records_by_name",
    "annotations_by_image",
    "coco",
    "image_dir",
    "model_v6b",
]:
    obj = globals().get(key)
    if obj is None:
        print(f"{key}: NOT LOADED")
    else:
        try:
            print(f"{key}: AVAILABLE ({len(obj)} entries)")
        except TypeError:
            print(f"{key}: AVAILABLE")

print("\nPASS: Environment checked.")

In [ ]:
from pathlib import Path

evaluation_names = [
    "20110421164914Mh.jpeg",
    "20220718185352Mh.jpeg",
    "20111226155454Bh.jpeg",
    "20141007224934Lh.jpeg",
    "20160726135354Bh.jpeg",
    "20130502063134Lh.jpeg",
    "20110823120334Ch.jpeg",
    "20140116180214Mh.jpeg",
    "20130510134654Bh.jpeg",
    "20150112081954Uh.jpeg",
]

# Verify that all photographs belong to our validation split
validation_set = set(val_names)

missing_split = [
    name for name in evaluation_names
    if name not in validation_set
]

missing_files = [
    name for name in evaluation_names
    if not (Path(image_dir) / name).is_file()
]

print("V6-B EVALUATION RESTORATION")
print("-" * 45)
print("Evaluation photographs:", len(evaluation_names))
print("Validation split:", len(val_names))
print("Not in validation split:", missing_split)
print("Missing image files:", missing_files)

assert not missing_split, "Validation split mismatch."
assert not missing_files, "Some images are missing."

print("\nPASS: Original 10-image evaluation set restored.")

In [ ]:
import numpy as np
import torch
import cv2
from PIL import Image, ImageDraw
from scipy.optimize import linear_sum_assignment

# Restore annotation lookup independently of earlier notebook variables.
anns_by_id = {}
for ann in coco["annotations"]:
    anns_by_id.setdefault(str(ann["image_id"]), []).append(ann)

images_by_name = {}
for rec in coco["images"]:
    images_by_name.setdefault(rec["file_name"], []).append(rec)

def get_gt_masks(name, H, W):
    candidates = images_by_name[name]
    rec = sorted(
        candidates,
        key=lambda r: (
            -len(anns_by_id.get(str(r["id"]), [])),
            str(r["id"])
        )
    )[0]

    masks = []
    for ann in anns_by_id.get(str(rec["id"]), []):
        canvas = Image.new("L", (W, H), 0)
        draw = ImageDraw.Draw(canvas)

        for poly in ann.get("segmentation", []):
            pts = list(zip(poly[::2], poly[1::2]))
            if len(pts) >= 3:
                draw.polygon(pts, fill=1)

        m = np.asarray(canvas, dtype=bool)
        if m.any():
            masks.append(m)

    return masks

def tile_starts(size, tile=512, stride=256):
    starts = list(range(0, max(size - tile + 1, 1), stride))
    last = max(size - tile, 0)
    if not starts or starts[-1] != last:
        starts.append(last)
    return starts

def infer_full_image(rgb):
    H, W = rgb.shape[:2]
    sums = np.zeros((H, W), np.float32)
    counts = np.zeros((H, W), np.float32)

    model_v6b.eval()
    with torch.inference_mode():
        for top in tile_starts(H):
            for left in tile_starts(W):
                crop = rgb[top:top+512, left:left+512]
                x = torch.from_numpy(
                    np.ascontiguousarray(crop.transpose(2, 0, 1))
                ).unsqueeze(0).float().to(device) / 255.0

                if device.type == "cuda":
                    with torch.autocast("cuda", dtype=torch.float16):
                        output = model_v6b(x)
                else:
                    output = model_v6b(x)

                p = torch.sigmoid(output)[0, 0].float().cpu().numpy()
                h, w = p.shape
                sums[top:top+h, left:left+w] += p
                counts[top:top+h, left:left+w] += 1

    return sums / np.maximum(counts, 1)

def extract_instances(prob, threshold):
    binary = (prob >= threshold).astype(np.uint8)
    kernel = np.ones((3, 3), np.uint8)

    clean = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    clean = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, kernel)

    count, lab, stats, _ = cv2.connectedComponentsWithStats(
        clean, connectivity=8
    )

    predictions = []
    for i in range(1, count):
        if stats[i, cv2.CC_STAT_AREA] < 500:
            continue
        mask = lab == i
        if float(prob[mask].mean()) >= 0.80:
            predictions.append(mask)

    return predictions

def instance_metrics(predictions, ground_truth):
    if not predictions or not ground_truth:
        return (0, len(predictions), len(ground_truth), 0.0)

    ious = np.zeros((len(ground_truth), len(predictions)), np.float32)

    for i, gt in enumerate(ground_truth):
        for j, pred in enumerate(predictions):
            intersection = np.count_nonzero(gt & pred)
            if intersection:
                union = np.count_nonzero(gt | pred)
                ious[i, j] = intersection / union

    # One-to-one matching with IoU >= 0.50.
    eligible = ious >= 0.50
    reward = np.where(eligible, 2.0 + ious, 0.0)
    rows, cols = linear_sum_assignment(-reward)

    matched = [
        float(ious[i, j])
        for i, j in zip(rows, cols)
        if eligible[i, j]
    ]

    tp = len(matched)
    return (
        tp,
        len(predictions) - tp,
        len(ground_truth) - tp,
        sum(matched)
    )

totals = {
    0.35: np.zeros(4, dtype=np.float64),
    0.55: np.zeros(4, dtype=np.float64)
}

print("V6-B — FULL-IMAGE THRESHOLD COMPARISON")
print("-" * 72)
print("Image | GT | PQ @ 0.35 | PQ @ 0.55 | Better")
print("-" * 72)

for idx, name in enumerate(evaluation_names, start=1):
    rgb = np.asarray(Image.open(image_dir / name).convert("RGB"))
    H, W = rgb.shape[:2]

    gt = get_gt_masks(name, H, W)
    prob_map = infer_full_image(rgb)

    image_pq = {}

    for threshold in [0.35, 0.55]:
        pred = extract_instances(prob_map, threshold)
        tp, fp, fn, sum_iou = instance_metrics(pred, gt)

        totals[threshold] += [tp, fp, fn, sum_iou]
        denom = tp + 0.5 * (fp + fn)
        image_pq[threshold] = sum_iou / denom if denom else 0.0

    better = (
        "0.35" if image_pq[0.35] > image_pq[0.55]
        else "0.55" if image_pq[0.55] > image_pq[0.35]
        else "Tie"
    )

    print(
        f"{idx:02d}/10 | {len(gt):2d} | "
        f"{image_pq[0.35]:8.4f} | "
        f"{image_pq[0.55]:8.4f} | {better}",
        flush=True
    )

print("\nAGGREGATED RESULTS")
print("-" * 55)

for threshold in [0.35, 0.55]:
    tp, fp, fn, sum_iou = totals[threshold]
    denom = tp + 0.5 * (fp + fn)
    pq = sum_iou / denom if denom else 0.0

    print(
        f"Threshold {threshold:.2f}: "
        f"TP={int(tp)}, FP={int(fp)}, FN={int(fn)}, PQ={pq:.4f}"
    )

print("\nOriginal V6-B reference: PQ=0.3302")
print("PASS: Ten-image threshold comparison completed.")

In [ ]:
print("V6-B — COMPONENT FRAGMENTATION DIAGNOSTIC")
print("-" * 68)
print("Threshold | Components | Area >= 500 | Final retained")

for threshold in [0.35, 0.55]:
    binary_t = (prob_map >= threshold).astype(np.uint8)
    kernel = np.ones((3, 3), dtype=np.uint8)

    cleaned = cv2.morphologyEx(
        binary_t, cv2.MORPH_OPEN, kernel
    )
    cleaned = cv2.morphologyEx(
        cleaned, cv2.MORPH_CLOSE, kernel
    )

    count, component_labels, component_stats, _ = (
        cv2.connectedComponentsWithStats(
            cleaned, connectivity=8
        )
    )

    total_components = count - 1

    area_qualified = [
        i for i in range(1, count)
        if component_stats[i, cv2.CC_STAT_AREA] >= 500
    ]

    retained = [
        i for i in area_qualified
        if float(
            prob_map[component_labels == i].mean()
        ) >= 0.80
    ]

    print(
        f"   {threshold:.2f}   | "
        f"{total_components:10d} | "
        f"{len(area_qualified):11d} | "
        f"{len(retained):14d}"
    )

print("\nPASS: Component fragmentation diagnostic completed.")

In [ ]:
print("V6-B — THRESHOLD × CONFIDENCE DIAGNOSTIC")
print("-" * 65)
print("Threshold | Confidence | Pred | TP | FP | FN | PQ")

for threshold in [0.35, 0.55]:
    binary_t = (prob_map >= threshold).astype(np.uint8)
    kernel = np.ones((3, 3), np.uint8)

    clean_t = cv2.morphologyEx(
        binary_t, cv2.MORPH_OPEN, kernel
    )
    clean_t = cv2.morphologyEx(
        clean_t, cv2.MORPH_CLOSE, kernel
    )

    count, lab, st, _ = cv2.connectedComponentsWithStats(
        clean_t, connectivity=8
    )

    # Compute candidate components once per threshold.
    candidates = []
    for j in range(1, count):
        if st[j, cv2.CC_STAT_AREA] < 500:
            continue

        mask = lab == j
        candidates.append((mask, float(prob_map[mask].mean())))

    for confidence in [0.0, 0.70, 0.80, 0.90]:
        predictions = [
            mask for mask, score in candidates
            if score >= confidence
        ]

        tp, fp, fn, sum_iou = instance_metrics(
            predictions, gt
        )

        denominator = tp + 0.5 * (fp + fn)
        pq = sum_iou / denominator if denominator else 0.0

        print(
            f"   {threshold:.2f}   |    {confidence:.2f}    | "
            f"{len(predictions):4d} | {tp:2d} | "
            f"{fp:2d} | {fn:2d} | {pq:.4f}"
        )

print("\nPASS: Threshold-confidence interaction evaluated.")

In [ ]:
import json
from pathlib import Path

original_set = set(evaluation_names)

remaining_names = sorted(
    set(val_names) - original_set
)

assert len(remaining_names) == 132
assert len(set(val_names)) == 142

# Deterministic selection, established before inspecting predictions.
import numpy as np
rng = np.random.default_rng(2026)
diagnostic_names = sorted(
    rng.choice(remaining_names, size=30, replace=False).tolist()
)

assert not (set(diagnostic_names) & original_set)
assert all((Path(image_dir) / n).is_file() for n in diagnostic_names)

manifest = {
    "purpose": "V6-B threshold generalization test",
    "selection_seed": 2026,
    "original_exploratory_images": sorted(original_set),
    "new_diagnostic_images": diagnostic_names,
    "frozen_comparisons": [
        {"threshold": 0.55, "min_area": 500, "confidence": 0.80},
        {"threshold": 0.35, "min_area": 500, "confidence": 0.80}
    ]
}

output = Path("/kaggle/working/v6b_diagnostic_manifest.json")
output.write_text(json.dumps(manifest, indent=2))

print("V6-B — NEW EVALUATION SUBSET")
print("-" * 45)
print("Previously used images:", len(original_set))
print("Available unused images:", len(remaining_names))
print("New diagnostic images:", len(diagnostic_names))
print("Overlap:", len(set(diagnostic_names) & original_set))
print("Manifest:", output)
print("\nPASS: Independent diagnostic subset selected.")

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path

# Confirm the functions and fixed diagnostic sample are available
assert len(diagnostic_names) == 30
assert not (set(diagnostic_names) & set(evaluation_names))

assert callable(infer_full_image)
assert callable(get_gt_masks)
assert callable(extract_instances)
assert callable(instance_metrics)

thresholds = [0.35, 0.55]

totals_30 = {
    t: np.zeros(4, dtype=np.float64)
    for t in thresholds
}

per_image_30 = []

print("V6-B — 30-IMAGE GENERALIZATION TEST")
print("-" * 75)
print("Image | GT | PQ @ 0.35 | PQ @ 0.55 | Better")
print("-" * 75)

for idx, name in enumerate(diagnostic_names, start=1):

    with Image.open(Path(image_dir) / name) as im:
        rgb = np.asarray(im.convert("RGB"))

    H, W = rgb.shape[:2]

    ground_truth = get_gt_masks(name, H, W)

    # One neural-network inference per image
    probability = infer_full_image(rgb)

    scores = {}

    for threshold in thresholds:

        predictions = extract_instances(
            probability, threshold
        )

        tp, fp, fn, sum_iou = instance_metrics(
            predictions, ground_truth
        )

        totals_30[threshold] += [
            tp, fp, fn, sum_iou
        ]

        denominator = tp + 0.5 * (fp + fn)
        scores[threshold] = (
            sum_iou / denominator
            if denominator > 0 else 0.0
        )

    better = (
        "0.35" if scores[0.35] > scores[0.55]
        else "0.55" if scores[0.55] > scores[0.35]
        else "Tie"
    )

    per_image_30.append({
        "name": name,
        "gt": len(ground_truth),
        "pq_035": scores[0.35],
        "pq_055": scores[0.55],
        "better": better
    })

    print(
        f"{idx:02d}/30 | "
        f"{len(ground_truth):2d} | "
        f"{scores[0.35]:8.4f} | "
        f"{scores[0.55]:8.4f} | "
        f"{better}",
        flush=True
    )

print("\nAGGREGATED RESULTS")
print("-" * 60)

for threshold in thresholds:

    tp, fp, fn, sum_iou = totals_30[threshold]

    denominator = tp + 0.5 * (fp + fn)

    pq = (
        sum_iou / denominator
        if denominator > 0 else 0.0
    )

    print(
        f"Threshold {threshold:.2f}: "
        f"TP={int(tp)}, "
        f"FP={int(fp)}, "
        f"FN={int(fn)}, "
        f"PQ={pq:.4f}"
    )

wins_035 = sum(r["better"] == "0.35" for r in per_image_30)
wins_055 = sum(r["better"] == "0.55" for r in per_image_30)
ties = sum(r["better"] == "Tie" for r in per_image_30)

print("\nIMAGE-LEVEL COMPARISON")
print("0.35 better:", wins_035)
print("0.55 better:", wins_055)
print("Ties:", ties)

# Save numerical results for later research analysis.
import json

output = Path(
    "/kaggle/working/v6b_threshold_30image_results.json"
)

output.write_text(json.dumps({
    "description": "Frozen V6-B threshold comparison",
    "images": diagnostic_names,
    "settings": {
        "thresholds": thresholds,
        "minimum_area": 500,
        "minimum_confidence": 0.80
    },
    "totals": {
        str(t): totals_30[t].tolist()
        for t in thresholds
    },
    "per_image": per_image_30
}, indent=2))

print("\nResults saved:", output)
print("PASS: 30-image generalization test completed.")

In [ ]:
print("V6-B+ — TRAINING READINESS")
print("-" * 50)

names = [
    "model_v6b",
    "train_names",
    "val_names",
    "train_loader",
    "loader_v6b",
    "optimizer_v6b",
    "scaler_v6b",
    "loss_v6b",
    "records_by_name",
    "annotations_by_image",
    "image_dir"
]

for name in names:
    obj = globals().get(name)

    if obj is None:
        print(f"{name:25s} NOT LOADED")
    else:
        try:
            detail = f"{len(obj)} entries"
        except (TypeError, AttributeError):
            detail = type(obj).__name__

        print(f"{name:25s} AVAILABLE — {detail}")

print("\nCheckpoint preserved:", 
      __import__("pathlib").Path(
          "/kaggle/working/v6b_final_research/"
          "v6b_convnext_unet_best.pth"
      ).exists())

print("PASS: Training environment inspected.")

In [ ]:
import inspect
import torch
from pathlib import Path

print("V6-B+ — ORIGINAL TRAINING CODE INSPECTION")
print("-" * 60)

# Find existing dataset, loss and training functions/classes.
candidates = []

for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue

    if not (
        inspect.isclass(obj)
        or inspect.isfunction(obj)
    ):
        continue

    key = name.lower()

    if any(term in key for term in [
        "dataset", "crop", "loss", "dice",
        "train", "augment", "loader"
    ]):
        candidates.append((name, obj))

for name, obj in sorted(candidates):
    print(f"{name:35s} {type(obj).__name__}")

print("\nCHECKPOINT")
checkpoint = Path(
    "/kaggle/working/v6b_final_research/"
    "v6b_convnext_unet_best.pth"
)

state = torch.load(
    checkpoint,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint type:", type(state).__name__)
print("First keys:", list(state.keys())[:5])
print("Checkpoint size:", round(
    checkpoint.stat().st_size / 1024**2, 2
), "MB")

print("\nCUDA:", torch.cuda.is_available())
print("PASS: Training code inspection completed.")

In [ ]:
from IPython import get_ipython
import inspect

ip = get_ipython()

print("V6-B — TRAINING CODE RECOVERY")
print("-" * 60)

print("\nDATASET CONSTRUCTOR:")
print(inspect.signature(SolarFilamentDatasetV6B))

print("\nLOSS FUNCTION SIGNATURE:")
print(inspect.signature(segmentation_loss))

history = list(ip.history_manager.get_range(output=False))

keywords = [
    "class SolarFilamentDatasetV6B",
    "def segmentation_loss",
    "optimizer_v6b",
]

found = set()

for _, _, source in history:
    for keyword in keywords:
        if keyword in source and keyword not in found:
            # Skip cells that only search for these keywords.
            if (
                "class SolarFilamentDatasetV6B" in source
                and source.lstrip().startswith("class ")
            ) or (
                "def segmentation_loss" in source
                and source.lstrip().startswith("def ")
            ) or (
                "optimizer_v6b = " in source
            ):
                print(f"\nFOUND: {keyword}")
                print(source[:12000])
                found.add(keyword)

print("\nRecovered:", sorted(found))
print("PASS: Notebook history checked.")

In [ ]:
from collections import defaultdict

print("V6-B+ — REPAIR TRAINING ANNOTATION LOOKUP")
print("-" * 60)

# Group annotations by their original image-record ID.
annotations_by_image = defaultdict(list)

for ann in coco["annotations"]:
    annotations_by_image[ann["image_id"]].append(ann)

# Group the multiple expert records for each physical photograph.
candidates_by_name = defaultdict(list)

for record in coco["images"]:
    candidates_by_name[record["file_name"]].append(record)

# Reconstruct the original deterministic expert-record selection.
records_by_name = {}

for filename, candidates in candidates_by_name.items():
    selected = sorted(
        candidates,
        key=lambda r: (
            -len(annotations_by_image.get(r["id"], [])),
            str(r["id"])
        )
    )[0]

    records_by_name[filename] = selected

# Verify complete coverage of our training and validation splits.
all_names = set(train_names) | set(val_names)

missing = all_names - set(records_by_name)

print("Unique photographs indexed:", len(records_by_name))
print("Training photographs:", len(train_names))
print("Validation photographs:", len(val_names))
print("Missing photographs:", len(missing))

assert not missing, f"Missing filenames: {list(missing)[:5]}"

# Test the original dataset class again.
train_dataset_v6b = SolarFilamentDatasetV6B(
    train_names,
    samples_per_image=4
)

train_loader_v6b = DataLoader(
    train_dataset_v6b,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

images, masks = next(iter(train_loader_v6b))

print("\nTRAINING BATCH")
print("Images:", tuple(images.shape))
print("Masks:", tuple(masks.shape))
print("Image range:", float(images.min()), float(images.max()))
print("Mask range:", float(masks.min()), float(masks.max()))

assert images.shape == (4, 3, 512, 512)
assert masks.shape == (4, 1, 512, 512)

print("\nPASS: V6-B training loader restored.")

In [ ]:
import torch
from pathlib import Path

print("V6-B+1 — TRAINING CONFIGURATION")
print("-" * 55)

# Confirm the frozen baseline is still available.
baseline_path = Path(
    "/kaggle/working/v6b_final_research/"
    "v6b_convnext_unet_best.pth"
)
assert baseline_path.is_file()

# Separate directory: never overwrite the baseline.
experiment_dir = Path("/kaggle/working/v6b_plus1")
experiment_dir.mkdir(parents=True, exist_ok=True)

# Reload the exact epoch-3 weights before the experiment.
state = torch.load(
    baseline_path,
    map_location="cpu",
    weights_only=True
)
model_v6b.load_state_dict(state, strict=True)
model_v6b.to(device)

# Keep the original differential learning rates.
encoder_params = list(model_v6b.encoder.parameters())

decoder_params = [
    p for name, p in model_v6b.named_parameters()
    if not name.startswith("encoder.")
]

optimizer_v6b = torch.optim.AdamW(
    [
        {"params": encoder_params, "lr": 1e-5},
        {"params": decoder_params, "lr": 1e-4},
    ],
    weight_decay=1e-4
)

scaler_v6b = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)

# Verify the original loss accepts model outputs and masks.
images, masks = next(iter(train_loader_v6b))

model_v6b.eval()
with torch.no_grad():
    x = images[:1].to(device)
    y = masks[:1].to(device)

    logits = model_v6b(x)
    test_loss = segmentation_loss(logits, y)

assert logits.shape == y.shape
assert torch.isfinite(test_loss).all()

print("Starting checkpoint: epoch 3")
print("Training photographs:", len(train_names))
print("Samples per epoch:", len(train_dataset_v6b))
print("Batches per epoch:", len(train_loader_v6b))
print("Encoder LR:", optimizer_v6b.param_groups[0]["lr"])
print("Decoder LR:", optimizer_v6b.param_groups[1]["lr"])
print("Loss test:", round(float(test_loss.item()), 4))
print("Experiment directory:", experiment_dir)

print("\nPASS: V6-B+1 configured without training.")

In [ ]:
import torch
import numpy as np
from pathlib import Path

model_v6b.train()

running_loss = 0.0
num_batches = 0

print("V6-B+1 — WARM-START TRAINING")
print("-" * 55)
print("Starting from: original epoch 3")
print("Training: one additional epoch")
print("Batches:", len(train_loader_v6b))

for step, (images, masks) in enumerate(train_loader_v6b, start=1):

    images = images.to(device, non_blocking=True)
    masks = masks.to(device, non_blocking=True)

    optimizer_v6b.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16 if device.type == "cuda" else torch.bfloat16,
        enabled=(device.type == "cuda")
    ):
        logits = model_v6b(images)
        loss = segmentation_loss(logits, masks)

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Non-finite training loss at batch {step}"
        )

    scaler_v6b.scale(loss).backward()
    scaler_v6b.step(optimizer_v6b)
    scaler_v6b.update()

    running_loss += float(loss.detach().item())
    num_batches += 1

    if step % 100 == 0 or step == len(train_loader_v6b):
        print(
            f"Batch {step:3d}/{len(train_loader_v6b)} | "
            f"Mean loss: {running_loss / num_batches:.4f}",
            flush=True
        )

epoch_loss = running_loss / num_batches

checkpoint_plus1 = (
    experiment_dir / "v6b_plus1_warmstart_epoch4.pth"
)

torch.save(model_v6b.state_dict(), checkpoint_plus1)

model_v6b.eval()

print("\nV6-B+1 — EPOCH 4 COMPLETE")
print("-" * 55)
print("Mean training loss:", round(epoch_loss, 4))
print("Checkpoint:", checkpoint_plus1)
print(
    "Checkpoint size:",
    round(checkpoint_plus1.stat().st_size / 1024**2, 2),
    "MB"
)
print("Original epoch-3 checkpoint: unchanged")
print("PASS: Warm-start epoch 4 completed.")

In [ ]:
import numpy as np
import torch
from pathlib import Path
from PIL import Image

print("V6-B+1 — EPOCH 4 INSTANCE EVALUATION")
print("-" * 60)

checkpoint_plus1 = Path(
    "/kaggle/working/v6b_plus1/"
    "v6b_plus1_warmstart_epoch4.pth"
)

assert checkpoint_plus1.is_file()

# Ensure the model contains our new epoch-4 weights.
model_v6b.load_state_dict(
    torch.load(
        checkpoint_plus1,
        map_location="cpu",
        weights_only=True
    ),
    strict=True
)
model_v6b.to(device)
model_v6b.eval()

totals_epoch4 = np.zeros(4, dtype=np.float64)

print("Image | GT | Pred | TP | FP | FN | PQ")
print("-" * 60)

for idx, name in enumerate(evaluation_names, start=1):

    with Image.open(Path(image_dir) / name) as im:
        rgb = np.asarray(im.convert("RGB"))

    H, W = rgb.shape[:2]

    gt = get_gt_masks(name, H, W)

    # Reuse our verified full-resolution inference pipeline.
    probability = infer_full_image(rgb)

    predictions = extract_instances(
        probability,
        threshold=0.55
    )

    tp, fp, fn, sum_iou = instance_metrics(
        predictions, gt
    )

    totals_epoch4 += [tp, fp, fn, sum_iou]

    denominator = tp + 0.5 * (fp + fn)
    pq = sum_iou / denominator if denominator else 0.0

    print(
        f"{idx:02d}/10 | "
        f"{len(gt):2d} | "
        f"{len(predictions):4d} | "
        f"{tp:2d} | {fp:2d} | {fn:2d} | "
        f"{pq:.4f}",
        flush=True
    )

tp, fp, fn, sum_iou = totals_epoch4
denominator = tp + 0.5 * (fp + fn)
pq_epoch4 = sum_iou / denominator if denominator else 0.0

print("\nAGGREGATED RESULTS")
print("-" * 60)
print(f"Epoch 3 baseline: PQ = 0.3302")
print(
    f"Epoch 4 warm-start: TP={int(tp)}, "
    f"FP={int(fp)}, FN={int(fn)}, PQ={pq_epoch4:.4f}"
)
print(f"PQ difference: {pq_epoch4 - 0.3302:+.4f}")

print("\nPASS: Epoch-4 evaluation completed.")

In [ ]:
import torch

assert checkpoint_plus1.is_file()
assert len(train_loader_v6b) == 565

model_v6b.train()
running_loss = 0.0

print("V6-B+1 — WARM-START EPOCH 5")
print("-" * 55)

for step, (images, masks) in enumerate(train_loader_v6b, start=1):
    images = images.to(device, non_blocking=True)
    masks = masks.to(device, non_blocking=True)

    optimizer_v6b.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):
        logits = model_v6b(images)
        loss = segmentation_loss(logits, masks)

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at batch {step}")

    scaler_v6b.scale(loss).backward()
    scaler_v6b.step(optimizer_v6b)
    scaler_v6b.update()

    running_loss += loss.detach().item()

    if step % 100 == 0 or step == len(train_loader_v6b):
        print(
            f"Batch {step}/{len(train_loader_v6b)} | "
            f"Mean loss: {running_loss / step:.4f}",
            flush=True
        )

epoch5_loss = running_loss / len(train_loader_v6b)

epoch5_path = experiment_dir / "v6b_plus1_warmstart_epoch5.pth"

torch.save(model_v6b.state_dict(), epoch5_path)
model_v6b.eval()

print("\nEPOCH 5 COMPLETE")
print("Mean loss:", round(epoch5_loss, 4))
print("Checkpoint:", epoch5_path)
print("Epoch 3 and epoch 4 checkpoints remain unchanged.")
print("PASS: V6-B+1 epoch 5 saved.")

In [ ]:
import numpy as np
import torch
from pathlib import Path
from PIL import Image

epoch5_path = Path(
    "/kaggle/working/v6b_plus1/"
    "v6b_plus1_warmstart_epoch5.pth"
)

assert epoch5_path.is_file()

model_v6b.load_state_dict(
    torch.load(epoch5_path, map_location="cpu", weights_only=True),
    strict=True
)
model_v6b.to(device)
model_v6b.eval()

totals_epoch5 = np.zeros(4, dtype=np.float64)

print("V6-B+1 — WARM-START EPOCH 5 EVALUATION")
print("-" * 62)
print("Image | GT | Pred | TP | FP | FN | PQ")

for idx, name in enumerate(evaluation_names, start=1):
    with Image.open(Path(image_dir) / name) as im:
        rgb = np.asarray(im.convert("RGB"))

    H, W = rgb.shape[:2]
    ground_truth = get_gt_masks(name, H, W)

    probability = infer_full_image(rgb)
    predictions = extract_instances(probability, threshold=0.55)

    tp, fp, fn, sum_iou = instance_metrics(
        predictions, ground_truth
    )

    totals_epoch5 += [tp, fp, fn, sum_iou]

    denominator = tp + 0.5 * (fp + fn)
    pq = sum_iou / denominator if denominator else 0.0

    print(
        f"{idx:02d}/10 | {len(ground_truth):2d} | "
        f"{len(predictions):4d} | {tp:2d} | "
        f"{fp:2d} | {fn:2d} | {pq:.4f}",
        flush=True
    )

tp, fp, fn, sum_iou = totals_epoch5
denominator = tp + 0.5 * (fp + fn)
pq_epoch5 = sum_iou / denominator if denominator else 0.0

print("\nCHECKPOINT COMPARISON — ORIGINAL 10 IMAGES")
print("-" * 62)
print("Original epoch 3:       PQ = 0.3302")
print("Warm-start epoch 4:     PQ = 0.3430")
print(f"Warm-start epoch 5:     PQ = {pq_epoch5:.4f}")

print(
    f"\nEpoch 5: TP={int(tp)}, FP={int(fp)}, FN={int(fn)}"
)
print(f"Change vs epoch 4: {pq_epoch5 - 0.3430:+.4f}")
print("PASS: Warm-start epoch 5 evaluated.")

In [ ]:
import json
import numpy as np
import torch
from pathlib import Path
from PIL import Image

print("V6-B+1 — 30-IMAGE CHECKPOINT COMPARISON")
print("-" * 66)

baseline_path = Path(
    "/kaggle/working/v6b_final_research/"
    "v6b_convnext_unet_best.pth"
)

epoch5_path = Path(
    "/kaggle/working/v6b_plus1/"
    "v6b_plus1_warmstart_epoch5.pth"
)

assert baseline_path.is_file() and epoch5_path.is_file()
assert len(diagnostic_names) == 30

checkpoints = {
    "Original epoch 3": baseline_path,
    "Warm-start epoch 5": epoch5_path,
}

comparison = {}

for label, path in checkpoints.items():
    model_v6b.load_state_dict(
        torch.load(path, map_location="cpu", weights_only=True),
        strict=True
    )
    model_v6b.to(device).eval()

    totals = np.zeros(4, dtype=np.float64)
    per_image = []

    print(f"\nEvaluating: {label}", flush=True)

    for idx, name in enumerate(diagnostic_names, start=1):
        with Image.open(Path(image_dir) / name) as im:
            rgb = np.asarray(im.convert("RGB"))

        H, W = rgb.shape[:2]
        gt = get_gt_masks(name, H, W)

        probability = infer_full_image(rgb)
        predictions = extract_instances(
            probability, threshold=0.55
        )

        tp, fp, fn, sum_iou = instance_metrics(
            predictions, gt
        )

        totals += [tp, fp, fn, sum_iou]
        denom = tp + 0.5 * (fp + fn)
        pq = sum_iou / denom if denom else 0.0

        per_image.append({
            "name": name,
            "gt": len(gt),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "pq": pq
        })

        print(
            f"{idx:02d}/30 | TP={tp:2d} FP={fp:2d} "
            f"FN={fn:2d} PQ={pq:.4f}",
            flush=True
        )

    tp, fp, fn, sum_iou = totals
    denom = tp + 0.5 * (fp + fn)
    aggregate_pq = sum_iou / denom if denom else 0.0

    comparison[label] = {
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "sum_iou": float(sum_iou),
        "PQ": float(aggregate_pq),
        "per_image": per_image
    }

print("\nAGGREGATED CHECKPOINT COMPARISON")
print("-" * 66)

for label, result in comparison.items():
    print(
        f"{label}: TP={result['TP']} "
        f"FP={result['FP']} FN={result['FN']} "
        f"PQ={result['PQ']:.4f}"
    )

difference = (
    comparison["Warm-start epoch 5"]["PQ"]
    - comparison["Original epoch 3"]["PQ"]
)
print(f"\nPQ difference: {difference:+.4f}")

output = Path(
    "/kaggle/working/v6b_plus1/"
    "epoch3_vs_epoch5_30images.json"
)
output.write_text(json.dumps(comparison, indent=2))

print("Results saved:", output)
print("PASS: Fixed-threshold checkpoint comparison completed.")

In [ ]:
import torch
import numpy as np
from pathlib import Path
from PIL import Image

checkpoints = {
    "Original epoch 3": Path(
        "/kaggle/working/v6b_final_research/"
        "v6b_convnext_unet_best.pth"
    ),
    "Warm-start epoch 5": Path(
        "/kaggle/working/v6b_plus1/"
        "v6b_plus1_warmstart_epoch5.pth"
    )
}

results_dice = {}

print("V6-B — PIXEL SEGMENTATION COMPARISON")
print("-" * 58)

for label, path in checkpoints.items():
    model_v6b.load_state_dict(
        torch.load(path, map_location="cpu", weights_only=True),
        strict=True
    )
    model_v6b.to(device).eval()

    intersection_total = 0
    pred_total = 0
    gt_total = 0

    for idx, name in enumerate(diagnostic_names, 1):
        with Image.open(Path(image_dir) / name) as im:
            rgb = np.asarray(im.convert("RGB"))

        H, W = rgb.shape[:2]
        masks = get_gt_masks(name, H, W)

        gt_union = np.zeros((H, W), dtype=bool)
        for mask in masks:
            gt_union |= mask

        probability = infer_full_image(rgb)
        prediction = probability >= 0.55

        intersection_total += np.count_nonzero(
            prediction & gt_union
        )
        pred_total += np.count_nonzero(prediction)
        gt_total += np.count_nonzero(gt_union)

        if idx % 10 == 0:
            print(f"{label}: {idx}/30", flush=True)

    dice = (
        2 * intersection_total / (pred_total + gt_total)
        if pred_total + gt_total else 1.0
    )

    results_dice[label] = dice

print("\nGLOBAL PIXEL DICE — 30 IMAGES")
for label, dice in results_dice.items():
    print(f"{label}: {dice:.4f}")

print(
    "Dice difference:",
    f"{results_dice['Warm-start epoch 5'] - results_dice['Original epoch 3']:+.4f}"
)

print("\nInstance PQ reference:")
print("Epoch 3: 0.3238")
print("Epoch 5: 0.3142")
print("PASS: Pixel-versus-instance evaluation completed.")

In [ ]:
import numpy as np
import torch
from pathlib import Path
from PIL import Image

checkpoints = {
    "Epoch 3": Path(
        "/kaggle/working/v6b_final_research/"
        "v6b_convnext_unet_best.pth"
    ),
    "Warm-start epoch 5": Path(
        "/kaggle/working/v6b_plus1/"
        "v6b_plus1_warmstart_epoch5.pth"
    )
}

print("V6-B — FALSE-POSITIVE ERROR ANALYSIS")
print("-" * 66)

for label, path in checkpoints.items():

    model_v6b.load_state_dict(
        torch.load(path, map_location="cpu", weights_only=True),
        strict=True
    )
    model_v6b.to(device).eval()

    categories = {
        "No GT overlap": 0,
        "Some overlap, below 25%": 0,
        "25–49% predicted-pixel overlap": 0,
        "50%+ predicted-pixel overlap": 0,
    }
    total_fp = 0

    for name in diagnostic_names:
        with Image.open(Path(image_dir) / name) as im:
            rgb = np.asarray(im.convert("RGB"))

        H, W = rgb.shape[:2]
        gt = get_gt_masks(name, H, W)
        prob = infer_full_image(rgb)
        preds = extract_instances(prob, threshold=0.55)

        if not preds:
            continue

        gt_union = np.zeros((H, W), dtype=bool)
        for mask in gt:
            gt_union |= mask

        ious = np.zeros((len(gt), len(preds)), dtype=np.float32)

        for i, g in enumerate(gt):
            for j, p in enumerate(preds):
                inter = np.count_nonzero(g & p)
                if inter:
                    ious[i, j] = inter / np.count_nonzero(g | p)

        # Reproduce the matching policy used by instance_metrics.
        from scipy.optimize import linear_sum_assignment

        matched_pred = set()

        if ious.size:
            eligible = ious >= 0.50
            reward = np.where(eligible, 2.0 + ious, 0.0)
            rows, cols = linear_sum_assignment(-reward)

            matched_pred = {
                int(j) for i, j in zip(rows, cols)
                if eligible[i, j]
            }

        for j, pred in enumerate(preds):
            if j in matched_pred:
                continue

            total_fp += 1

            overlap = (
                np.count_nonzero(pred & gt_union)
                / max(np.count_nonzero(pred), 1)
            )

            if overlap == 0:
                categories["No GT overlap"] += 1
            elif overlap < 0.25:
                categories["Some overlap, below 25%"] += 1
            elif overlap < 0.50:
                categories["25–49% predicted-pixel overlap"] += 1
            else:
                categories["50%+ predicted-pixel overlap"] += 1

    print(f"\n{label} — total false positives: {total_fp}")
    for category, count in categories.items():
        print(f"  {category}: {count}")

print("\nPASS: False-positive overlap diagnostic completed.")

In [ ]:
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from scipy.optimize import linear_sum_assignment

baseline_path = Path(
    "/kaggle/working/v6b_final_research/"
    "v6b_convnext_unet_best.pth"
)

model_v6b.load_state_dict(
    torch.load(baseline_path, map_location="cpu", weights_only=True),
    strict=True
)
model_v6b.to(device).eval()

categories = {
    "No expert overlap": 0,
    "Overlap, best IoU < 0.30": 0,
    "Near miss: IoU 0.30–0.49": 0,
    "IoU >= 0.50 but unmatched": 0,
}

print("V6-B — INDIVIDUAL INSTANCE ERROR ANALYSIS")
print("-" * 60)

for idx, name in enumerate(diagnostic_names, start=1):
    with Image.open(Path(image_dir) / name) as im:
        rgb = np.asarray(im.convert("RGB"))

    H, W = rgb.shape[:2]
    ground_truth = get_gt_masks(name, H, W)

    probability = infer_full_image(rgb)
    predictions = extract_instances(probability, 0.55)

    ious = np.zeros(
        (len(ground_truth), len(predictions)),
        dtype=np.float32
    )

    for i, gt_mask in enumerate(ground_truth):
        for j, pred_mask in enumerate(predictions):
            intersection = np.count_nonzero(gt_mask & pred_mask)
            if intersection:
                union = np.count_nonzero(gt_mask | pred_mask)
                ious[i, j] = intersection / union

    matched_predictions = set()

    if ious.size:
        eligible = ious >= 0.50
        reward = np.where(eligible, 2.0 + ious, 0.0)
        rows, cols = linear_sum_assignment(-reward)

        matched_predictions = {
            int(j)
            for i, j in zip(rows, cols)
            if eligible[i, j]
        }

    for j in range(len(predictions)):
        if j in matched_predictions:
            continue

        best_iou = float(ious[:, j].max()) if len(ground_truth) else 0

        if best_iou == 0:
            categories["No expert overlap"] += 1
        elif best_iou < 0.30:
            categories["Overlap, best IoU < 0.30"] += 1
        elif best_iou < 0.50:
            categories["Near miss: IoU 0.30–0.49"] += 1
        else:
            categories["IoU >= 0.50 but unmatched"] += 1

    if idx % 10 == 0:
        print(f"Processed {idx}/30 photographs", flush=True)

print("\nFALSE-POSITIVE CLASSIFICATION")
print("-" * 60)

for category, count in categories.items():
    print(f"{category}: {count}")

print("Total:", sum(categories.values()))
assert sum(categories.values()) == 160

print("\nPASS: Individual-instance diagnostic completed.")

In [ ]:
import torch
import torch.nn.functional as F

# Experimental parameter, fixed before training
BOUNDARY_WEIGHT = 0.5

def boundary_aware_loss(logits, targets):
    """
    Original BCE + Dice loss plus boundary-weighted BCE.

    Targets: binary filament masks [B, 1, H, W]
    Logits:  raw network outputs [B, 1, H, W]
    """

    # Preserve our original segmentation objective
    original_loss = segmentation_loss(logits, targets)

    # Detect foreground boundaries using local morphological gradients.
    # Max pooling acts as dilation; negative max pooling as erosion.
    dilated = F.max_pool2d(
        targets, kernel_size=3, stride=1, padding=1
    )

    eroded = -F.max_pool2d(
        -targets, kernel_size=3, stride=1, padding=1
    )

    boundary = (dilated - eroded).clamp(0, 1)

    # Additional BCE emphasis at mask boundaries
    pixel_bce = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none"
    )

    boundary_loss = (
        (pixel_bce * boundary).sum()
        / boundary.sum().clamp_min(1.0)
    )

    total_loss = original_loss + BOUNDARY_WEIGHT * boundary_loss

    return total_loss, original_loss, boundary_loss


# Test on one training batch before modifying model weights
images, masks = next(iter(train_loader_v6b))

model_v6b.eval()

with torch.no_grad():
    x = images[:1].to(device)
    y = masks[:1].to(device)

    logits = model_v6b(x)

    total, original, boundary = boundary_aware_loss(
        logits, y
    )

assert torch.isfinite(total)
assert torch.isfinite(original)
assert torch.isfinite(boundary)

print("V6-B+2 — BOUNDARY LOSS VERIFICATION")
print("-" * 55)
print("Original BCE + Dice:", round(original.item(), 4))
print("Boundary loss:", round(boundary.item(), 4))
print("Combined loss:", round(total.item(), 4))
print("Boundary weight:", BOUNDARY_WEIGHT)
print("Model parameters unchanged:", True)
print("PASS: Boundary-aware objective verified.")

In [ ]:
import torch
import torch.nn.functional as F

BOUNDARY_WEIGHT = 0.5

def boundary_aware_loss(logits, targets):
    original_loss = segmentation_loss(logits, targets)

    dilated = F.max_pool2d(
        targets, kernel_size=3, stride=1, padding=1
    )
    eroded = -F.max_pool2d(
        -targets, kernel_size=3, stride=1, padding=1
    )

    boundary = (dilated - eroded).clamp(0, 1)

    bce = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none"
    )

    # Extra boundary weighting while normalizing by total weight.
    weights = 1.0 + BOUNDARY_WEIGHT * boundary

    weighted_bce = (
        (weights * bce).sum() / weights.sum().clamp_min(1.0)
    )

    ordinary_bce = bce.mean()

    # Keep the original BCE + Dice objective,
    # replacing only its BCE contribution.
    total_loss = original_loss - ordinary_bce + weighted_bce

    return total_loss, original_loss, weighted_bce


# Verify on the current model, without training.
images, masks = next(iter(train_loader_v6b))
model_v6b.eval()

with torch.no_grad():
    x = images[:1].to(device)
    y = masks[:1].to(device)

    logits = model_v6b(x)
    total, original, weighted = boundary_aware_loss(logits, y)

assert torch.isfinite(total)
assert torch.isfinite(original)
assert torch.isfinite(weighted)

print("V6-B+2 — NORMALIZED BOUNDARY LOSS")
print("-" * 55)
print("Original loss:", round(original.item(), 4))
print("Weighted BCE:", round(weighted.item(), 4))
print("Combined loss:", round(total.item(), 4))
print("Boundary weight:", BOUNDARY_WEIGHT)
print("PASS: Revised loss verified without training.")

In [ ]:
import torch
from pathlib import Path

print("V6-B+2 — BOUNDARY-AWARE TRAINING")
print("-" * 55)

baseline = Path(
    "/kaggle/working/v6b_final_research/"
    "v6b_convnext_unet_best.pth"
)

boundary_dir = Path("/kaggle/working/v6b_plus2")
boundary_dir.mkdir(parents=True, exist_ok=True)

# Restore frozen epoch-3 weights
model_v6b.load_state_dict(
    torch.load(baseline, map_location="cpu", weights_only=True),
    strict=True
)
model_v6b.to(device)

# New optimizer with the same settings as V6-B+1
optimizer_boundary = torch.optim.AdamW(
    [
        {"params": list(model_v6b.encoder.parameters()), "lr": 1e-5},
        {
            "params": [
                p for name, p in model_v6b.named_parameters()
                if not name.startswith("encoder.")
            ],
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

scaler_boundary = torch.amp.GradScaler(
    "cuda", enabled=(device.type == "cuda")
)

model_v6b.train()
total_loss = 0.0

for step, (images, masks) in enumerate(train_loader_v6b, 1):
    images = images.to(device, non_blocking=True)
    masks = masks.to(device, non_blocking=True)

    optimizer_boundary.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):
        logits = model_v6b(images)
        loss, _, _ = boundary_aware_loss(logits, masks)

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at batch {step}")

    scaler_boundary.scale(loss).backward()
    scaler_boundary.step(optimizer_boundary)
    scaler_boundary.update()

    total_loss += loss.detach().item()

    if step % 100 == 0 or step == len(train_loader_v6b):
        print(
            f"Batch {step}/{len(train_loader_v6b)} | "
            f"Mean loss: {total_loss / step:.4f}",
            flush=True
        )

boundary_epoch4 = (
    boundary_dir / "v6b_plus2_boundary_epoch4.pth"
)

torch.save(model_v6b.state_dict(), boundary_epoch4)
model_v6b.eval()

print("\nV6-B+2 — TRAINING COMPLETE")
print("Mean training loss:", round(total_loss / len(train_loader_v6b), 4))
print("Checkpoint:", boundary_epoch4)
print("Baseline checkpoint unchanged.")
print("PASS: Boundary-aware epoch 4 saved.")

In [ ]:
import torch
import numpy as np
import json
from pathlib import Path
from PIL import Image

checkpoints = {
    "Original epoch 3": Path(
        "/kaggle/working/v6b_final_research/"
        "v6b_convnext_unet_best.pth"
    ),
    "BCE+Dice epoch 4": Path(
        "/kaggle/working/v6b_plus1/"
        "v6b_plus1_warmstart_epoch4.pth"
    ),
    "Boundary epoch 4": Path(
        "/kaggle/working/v6b_plus2/"
        "v6b_plus2_boundary_epoch4.pth"
    ),
}

results_44 = {}

print("V6-B — BOUNDARY-LOSS COMPARISON")
print("-" * 70)

for label, path in checkpoints.items():
    assert path.is_file(), f"Missing checkpoint: {path}"

    model_v6b.load_state_dict(
        torch.load(path, map_location="cpu", weights_only=True),
        strict=True
    )
    model_v6b.to(device).eval()

    totals = np.zeros(4, dtype=np.float64)

    for name in evaluation_names:
        with Image.open(Path(image_dir) / name) as im:
            rgb = np.asarray(im.convert("RGB"))

        H, W = rgb.shape[:2]
        gt = get_gt_masks(name, H, W)

        probability = infer_full_image(rgb)
        predictions = extract_instances(
            probability, threshold=0.55
        )

        tp, fp, fn, sum_iou = instance_metrics(
            predictions, gt
        )

        totals += [tp, fp, fn, sum_iou]

    tp, fp, fn, sum_iou = totals
    denominator = tp + 0.5 * (fp + fn)
    pq = sum_iou / denominator if denominator else 0.0

    results_44[label] = {
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "sum_iou": float(sum_iou),
        "PQ": float(pq)
    }

    print(
        f"{label:22s} "
        f"TP={int(tp):2d} "
        f"FP={int(fp):2d} "
        f"FN={int(fn):2d} "
        f"PQ={pq:.4f}",
        flush=True
    )

output = Path(
    "/kaggle/working/v6b_plus2/"
    "boundary_vs_bce_10images.json"
)
output.write_text(json.dumps(results_44, indent=2))

print("\nBoundary vs ordinary BCE+Dice epoch 4:")
print(
    "PQ difference:",
    f"{results_44['Boundary epoch 4']['PQ'] - results_44['BCE+Dice epoch 4']['PQ']:+.4f}"
)

print("\nResults saved:", output)
print("PASS: Boundary-loss comparison completed.")